# B5: Dependency Parsing and Sentence Analysis

Parse sentences with a dependency parser, identify syntactic relationships (subject, object, modifiers), compare dependency and constituency parsing, visualize the parse trees, and analyse parsing errors and limitations.

**Dataset:** a custom set of 12 sentences, from simple to tricky

In [2]:
# spacy : the dependency parser and displaCy for drawing
# nltk  : a small grammar for the constituency tree
!pip install -q spacy nltk pandas
!python -m spacy download en_core_web_sm

In [3]:
import pandas as pd
import spacy
from spacy import displacy
import nltk
nlp = spacy.load("en_core_web_sm")

## Custom sentence dataset

In [4]:
sentences = [
    "The student submitted the assignment.",
    "Priya gave her brother a new laptop.",
    "The report was written by the committee.",
    "Rahul and Sneha bought vegetables at the market.",
    "The book that I borrowed from the library is very interesting.",
    "She wants to learn natural language processing.",
    "Although it was raining, the match continued.",
    "The chef cooked a delicious meal for the guests.",
    "I saw the man with the telescope.",
    "The old man the boat.",
    "The horse raced past the barn fell.",
    "pls send the notes asap bro"]
len(sentences)

12

## (a) Parse a sentence

In [5]:
doc = nlp(sentences[0])
pd.DataFrame([(t.i, t.text, t.pos_, t.dep_, t.head.text) for t in doc],
             columns=["id", "word", "POS", "relation", "head"])

,id,word,POS,relation,head
0,0,The,DET,det,student
1,1,student,NOUN,nsubj,submitted
2,2,submitted,VERB,ROOT,submitted
3,3,the,DET,det,assignment
4,4,assignment,NOUN,dobj,submitted
5,5,.,PUNCT,punct,submitted


In [6]:
for label in sorted({t.dep_ for t in doc}):
    print(f"{label:<6} {spacy.explain(label)}")

ROOT   root
det    determiner
dobj   direct object
nsubj  nominal subject
punct  punctuation


## (b) Subjects, objects and modifiers

In [7]:
SUBJ = {"nsubj", "nsubjpass", "csubj"}
OBJ = {"dobj", "dative", "pobj", "attr"}
MOD = {"amod", "advmod", "prep", "poss", "compound", "relcl", "acl", "neg", "nummod"}

def relations(sentence):
    doc = nlp(sentence)
    return {"subjects": [(t.text, t.head.text) for t in doc if t.dep_ in SUBJ],
            "objects": [(t.text, t.head.text) for t in doc if t.dep_ in OBJ],
            "modifiers": [(t.text, t.dep_, t.head.text) for t in doc if t.dep_ in MOD]}

relations("The chef cooked a delicious meal for the guests.")

{'subjects': [('chef', 'cooked')],
 'objects': [('meal', 'cooked'), ('guests', 'for')],
 'modifiers': [('delicious', 'amod', 'meal'), ('for', 'prep', 'meal')]}

In [8]:
pd.DataFrame([{"sentence": s, **relations(s)} for s in sentences[:8]])

,sentence,subjects,objects,modifiers
0,The student submitted the assignment.,"[(student, submitted)]","[(assignment, submitted)]",[]
1,Priya gave her brother a new laptop.,"[(Priya, gave)]","[(brother, gave), (laptop, gave)]","[(her, poss, brother), (new, amod, laptop)]"
2,The report was written by the committee.,"[(report, written)]","[(committee, by)]",[]
3,Rahul and Sneha bought vegetables at the market.,"[(Rahul, bought)]","[(vegetables, bought), (market, at)]","[(at, prep, bought)]"
4,The book that I borrowed from the library is v...,"[(book, is), (I, borrowed)]","[(that, borrowed), (library, from)]","[(borrowed, relcl, book), (from, prep, borrowe..."
5,She wants to learn natural language processing.,"[(She, wants)]","[(processing, learn)]","[(natural, amod, language), (language, compoun..."
6,"Although it was raining, the match continued.","[(it, raining), (match, continued)]",[],[]
7,The chef cooked a delicious meal for the guests.,"[(chef, cooked)]","[(meal, cooked), (guests, for)]","[(delicious, amod, meal), (for, prep, meal)]"


## (c) Dependency vs constituency parsing

Constituency parsing groups words into phrases (NP, VP, PP) using a grammar. A small grammar for one sentence:

In [9]:
grammar = nltk.CFG.fromstring("""
S -> NP VP
NP -> Det N | Det Adj N
VP -> V NP
Det -> 'The' | 'the'
Adj -> 'new'
N -> 'student' | 'assignment'
V -> 'submitted'
""")
tree = next(nltk.ChartParser(grammar).parse("The student submitted the assignment".split()))
tree.pretty_print()

                    S                       
      ______________|______                  
     |                     VP               
     |               ______|___              
     NP             |          NP           
  ___|_____         |       ___|______       
Det        N        V     Det         N     
 |         |        |      |          |      
The     student submitted the     assignment



In [10]:
# the same sentence as dependencies: one arrow per word
for t in nlp("The student submitted the assignment"):
    print(f"{t.text:<11} --{t.dep_}--> {t.head.text}")

The         --det--> student
student     --nsubj--> submitted
submitted   --ROOT--> submitted
the         --det--> assignment
assignment  --dobj--> submitted


| | Constituency | Dependency |
|---|---|---|
| Shows | which words form phrases | which word modifies which |
| Nodes | phrases (S, NP, VP) + words | words only |
| Labels | phrase types | relations (nsubj, dobj, amod) |
| Subject / object | implicit (NP under S) | explicit labels |
| Free word order (Hindi, Marathi) | harder | easier |
| Used for | grammar checking, linguistics | information extraction, question answering |

## (d) Visualize the parse

In [11]:
displacy.render(nlp(sentences[1]), style="dep", jupyter=True, options={"compact": True, "distance": 110})

In [12]:
displacy.render(nlp(sentences[4]), style="dep", jupyter=True, options={"compact": True, "distance": 100})

## (e) Parsing errors

In [13]:
def show(sentence):
    doc = nlp(sentence)
    root = [t for t in doc if t.dep_ == "ROOT"][0]
    print(sentence)
    print("   main verb (ROOT):", root.text, "| subject:", [t.text for t in doc if t.dep_ in SUBJ])
    print("   ", [(t.text, t.dep_, t.head.text) for t in doc])
    print()

for s in sentences[7:]:
    show(s)

The chef cooked a delicious meal for the guests.
   main verb (ROOT): cooked | subject: ['chef']
    [('The', 'det', 'chef'), ('chef', 'nsubj', 'cooked'), ('cooked', 'ROOT', 'cooked'), ('a', 'det', 'meal'), ('delicious', 'amod', 'meal'), ('meal', 'dobj', 'cooked'), ('for', 'prep', 'meal'), ('the', 'det', 'guests'), ('guests', 'pobj', 'for'), ('.', 'punct', 'cooked')]

I saw the man with the telescope.
   main verb (ROOT): saw | subject: ['I']
    [('I', 'nsubj', 'saw'), ('saw', 'ROOT', 'saw'), ('the', 'det', 'man'), ('man', 'dobj', 'saw'), ('with', 'prep', 'man'), ('the', 'det', 'telescope'), ('telescope', 'pobj', 'with'), ('.', 'punct', 'saw')]

The old man the boat.
   main verb (ROOT): man | subject: []
    [('The', 'det', 'man'), ('old', 'amod', 'man'), ('man', 'ROOT', 'man'), ('the', 'det', 'boat'), ('boat', 'appos', 'man'), ('.', 'punct', 'man')]

The horse raced past the barn fell.
   main verb (ROOT): raced | subject: ['horse']
    [('The', 'det', 'horse'), ('horse', 'nsubj', '

Check each sentence above against its correct reading:
- **"for the guests"**: the guests are who the cooking was for, so it should attach to *cooked*; a parser often attaches it to *meal*.
- **The telescope sentence** has two correct readings; the parser can only return one.
- **"The old man the boat"** means "old people crew the boat" (*man* is the verb). A parser trained on normal text reads *old man* as a noun phrase and cannot find a verb.
- **"The horse raced past the barn fell"** means "the horse that was raced past the barn fell" (*fell* is the main verb). Garden-path sentences like this mislead parsers, which decide word by word.
- **Informal text** ("pls ... asap bro") is unlike the training data, so words get odd labels.

## Limitations
- One tree per sentence, even when the sentence is ambiguous.
- Attachment of prepositional phrases needs meaning and world knowledge, which a syntactic parser does not have.
- Trained on edited news and web text: accuracy drops on chat, typos and long sentences.
- Errors pass on to everything that uses the parse (relation extraction, question answering).